# 02 · Temporal analysis (weekly time series)

Builds weekly distortion counts, normalises by weekly word volume, z-scores, 
and renders the small-multiples time-series figure — all via `src/` helpers.


In [ ]:
# Make `src` importable when running the notebook from the notebooks/ folder.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.config import load_config
cfg = load_config()
COMMUNITY = 'sgexams'  # change to 'teenagers' to analyse the other community

In [ ]:
from src.data.datasets import load_community
from src.cognitive_distortions.timeseries import weekly_distortion_counts
from src.temporal_analysis.timeseries_ops import normalize_by_volume, zscore_frame, VOLUME_COL
data = load_community(cfg, COMMUNITY)
ts = weekly_distortion_counts(data['comments'], cfg.get('data.columns.comment_text', ['body']))
norm = normalize_by_volume(ts, scale=cfg.get('temporal.normalize_scale', 100))
z = zscore_frame(norm)
z.head()

In [ ]:
from src.visualization.plots import timeseries_tiles
fig = timeseries_tiles(z, title=f'{COMMUNITY}: z-score time series', covid_marker=None)
fig

In [ ]:
# Spike detection (>1 SD above trailing 4-week mean) for one category.
from src.temporal_analysis.timeseries_ops import filter_and_identify_spikes, category_columns
col = category_columns(ts)[1]
filtered, spikes = filter_and_identify_spikes(ts[col].tolist(), window_size=cfg.get('temporal.spike_window', 4))
print(col, '-> spike weeks:', spikes)